# Agent Monitoring Setup (Gemini)
This notebook sets up a LangGraph React Agent powered by Google Gemini, equipped with Unity Catalog tools to send Microsoft Teams alerts upon pipeline failures.

In [ ]:
# 1. Environment Setup & Dependencies
# Uninstall conflicting packages first to prevent version mismatches
%pip uninstall -y langchain-google-genai langchain langchain-core

# Install perfectly aligned 0.2.x family
%pip install langchain==0.2.16 langchain-core==0.2.43 langchain-google-genai==1.0.8 databricks-langchain

dbutils.library.restartPython()

In [ ]:
import os
import json
import urllib.request
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.databricks import UCFunctionToolkit
from langgraph.prebuilt import create_react_agent

## Register Unity Catalog Tool

In [ ]:
# 2. Register Unity Catalog Function for Teams Alert
# Using spark.sql so this runs smoothly within the Python execution context
spark.sql("""
CREATE OR REPLACE FUNCTION agent_monitoring.murugaveltarun.send_teams_alert(
  alert_title STRING COMMENT 'The heading of the alert.',
  alert_message STRING COMMENT 'The detailed explanation generated by the Agent.',
  webhook_url STRING COMMENT 'The secure Microsoft Teams Webhook URL.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Sends a formatted alert message to a Microsoft Teams channel via a webhook. Called by the AI agent after investigating a pipeline failure.'
AS $$
  import json
  import urllib.request

  payload = {
      "title": alert_title,
      "text": alert_message
  }
  
  data = json.dumps(payload).encode('utf-8')
  req = urllib.request.Request(
      webhook_url, 
      data=data, 
      headers={'Content-Type': 'application/json'}
  )
  
  try:
      with urllib.request.urlopen(req) as response:
          if response.status == 200:
              return f"Success: Alert delivered. Status: {response.status}"
          else:
              return f"Failure: Could not deliver alert. Status: {response.status}"
  except Exception as e:
      return f"System Error: Details: {str(e)}"
$$
""")

print("UC Function 'send_teams_alert' registered successfully.")

## Initialize the Agent

In [ ]:
# 3. Fetch Secrets & Configure Environment
teams_webhook = dbutils.secrets.get(scope="synergech_monitoring", key="teams_webhook_url")
gemini_api_key = dbutils.secrets.get(scope="synergech_monitoring", key="google_api_key")

os.environ["GOOGLE_API_KEY"] = gemini_api_key

# 4. Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

# 5. Bind Unity Catalog Tool to the Agent
toolkit = UCFunctionToolkit(
    warehouse_id="1a2b3661fe300d416eebc",  # Make sure this Warehouse ID matches your SQL warehouse
    function_names=["agent_monitoring.murugaveltarun.send_teams_alert"]
)
tools = toolkit.get_tools()

# 6. Define System Prompt
system_prompt = f"""
You are the Synergech Pipeline Detective, an elite AI monitoring agent.
Your job is to analyze Databricks pipeline error logs and explain them in simple, non-technical terms.

When you receive an error log:
1. Identify the root cause of the failure.
2. Summarize the impact (e.g., "The Driver Liability Assessment failed to run").
3. Suggest a next step for the data engineering team.
4. ONLY after you have done this analysis, use your tool to send the alert to Microsoft Teams.

IMPORTANT SECURITY INSTRUCTION: 
When you invoke the send_teams_alert tool, you MUST use this exact webhook URL for the webhook_url parameter: {teams_webhook}
"""

# Assemble LangGraph React Agent
agent_executor = create_react_agent(llm, tools, state_modifier=system_prompt)
print("Agent successfully initialized using Google Gemini and LangGraph!")

## Simulate Pipeline Failure & Trigger Agent

In [ ]:
# 7. Simulate Execution
messy_error_log = """
Py4JJavaError: An error occurred while calling o112.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 14.0 failed 4 times. 
Caused by: java.lang.NullPointerException: Value at column 'Driver_Age' in table 'raw_telemetry' cannot be null.
Constraint validation failed at synergech.liability.assessment.run(liability_calc.py:45)
"""

print("Waking up Agent to investigate...")
response = agent_executor.invoke({
    "messages": [("user", f"Our pipeline just crashed. Here is the log: {messy_error_log}")]
})

print("\n--- Final Agent Output ---")
print(response["messages"][-1].content)